In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_payment_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "PaymentId	string,"
        #     "SourceSystem	string,"
        #     "PortfolioKey	string,"
        #     "PortfolioName	string,"
        #     "IngestionTimestamp	timestamp,"
        #     "BatchId	string,"
        #     "PaymentKey	string,"
        #     "PaymentUniqueNumber	string,"
        #     "ExposureKey	string,"
        #     "PaymentActualDate	date,"
        #     "PaymentRecordDate	timestamp,"
        #     "PaymentAmount	decimal(15,2),"
        #     "PaymentIntIdentification	string,"
        #     "PaymentCostAmount	decimal(15,2),"
        #     "CoverageAmount	double,"
        #     "DebtAmount	double,"
        #     "MinimumPayment	double,"
        #     "CurrencyCode	string,"
        #     "Currency	string,"
        #     "PaymentTypeCode	string,"
        #     "PaymentType	string,"
        #     "ReasonCode	string,"
        #     "TransactionReasonCode	string,"
        #     "TransactionReason	string,"
        #     "ReportingDate	string,"
        #     "PrincipalAmount	decimal(15,2),"
        #     "CommissionAmount	decimal(15,2),"
        #     "InterestAmount	decimal(15,2),"
        #     "ExpensesAmount	decimal(15,2),"
        #     "InterestURL	string,"
        #     "Createdby	string,"
        #     "IsDownPaymentFlag	boolean,"
        #     "IsReversedFlag	string,"
        #     "PaymentISCOL	string,"
        #     "PaymentRecordType	string,"
        #     "PaymentTypeGroup string,"
        #     "PaymentTypeEng string,"
        #     "BusinessDate date"
        # )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Payment"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "profits",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== params ==================

        params_extra_filters = None

        if params_extra_filters is not None:
            params_extra_filters.append(date_filter)
            params_final_filters = " AND ".join(params_extra_filters)
        else:
            params_final_filters = date_filter

        params_selected_fields = [
            "serial_num",
            "param_source",
            "param_type",
            "description",
        ]

        params = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_params",
            params_final_filters,
            params_selected_fields,
        )

        params = params.filter(
            (params.param_source == "GD") & (params.param_type == "EXTYP")
        )

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
# ================== paymenttypegroupmapping ==================

        paymenttypegroupmapping_extra_filters = None

        if paymenttypegroupmapping_extra_filters is not None:
            paymenttypegroupmapping_extra_filters.append(date_filter)
            paymenttypegroupmapping_final_filters = " AND ".join(paymenttypegroupmapping_extra_filters)
        else:
            paymenttypegroupmapping_final_filters = date_filter

        paymenttypegroupmapping_selected_fields = [
            "PaymentTypeGroup",
            "ProfitsPaymentType",
            "ProfitsPaymentTypeCode",
        ]

        paymenttypegroupmapping = load_landed_specificationparams_table( 
            spark,
            reference_zone,  # type: ignore
            "paymenttypegroupmapping",
            paymenttypegroupmapping_selected_fields,
        )

        # ================== payments ==================

        payments_extra_filters = None

        if payments_extra_filters is not None:
            payments_extra_filters.append(date_filter)
            payments_final_filters = " AND ".join(payments_extra_filters)
        else:
            payments_final_filters = date_filter

            payments_selected_fields = [
                "trx_unit",
                "trx_usr",
                "trx_sn",
                "unit_code",
                "Record_date",
                "ACCOUNT_NUMBER",
                "actual_date",
                "amount",
                "CURRENCY",
                "GD_REASON",
                "principal",
                "commission",
                "interest",
                "expenses",
                "INTEREST_URL",
                "isdownpayment",
                "REVERSED_FLG",
                "businessdate",
            ]

            payments = load_landed_delta_table(
                spark,
                catalog_landed,  # type: ignore
                "profits_payments",
                payments_final_filters,
                payments_selected_fields,
            )
     
        payments_portfolio_join_condition = (
            payments["unit_code"] == portfolio["unit_code_profits"]
        )

        payments_paymenttypegroupmapping_join_condition = (
            payments["GD_REASON"].cast("string") == paymenttypegroupmapping["ProfitsPaymentTypeCode"].cast("string")
        )

        payments_Params_join_condition = payments["GD_REASON"] == params["serial_num"]

        result_df = payments.join(
            portfolio,
            payments_portfolio_join_condition,
            "left",
        ).join(
            params,
            payments_Params_join_condition,
            "left",
        ).join(
            paymenttypegroupmapping,
            payments_paymenttypegroupmapping_join_condition,
            "left",
        )

        # ================== payment_profits_df ==================
        payment_profits_primary_key_fields = [
            payments.actual_date,
            payments.trx_unit,
            payments.trx_usr,
            payments.trx_sn,
            payments.GD_REASON,
            lit("profits"),
        ]

        payment_profits_df = (
            result_df.withColumn(
                "PaymentId",
                sha2(concat_ws(";", *payment_profits_primary_key_fields), 256),
            )
            .withColumn(
                "PaymentKey",
                (concat_ws(";", *payment_profits_primary_key_fields)).cast("string"),
            )
            .withColumn(
                "PaymentUniqueNumber",
                (concat_ws(";", *payment_profits_primary_key_fields).cast("string")),
            )
            .withColumn(
                "PaymentIntIdentification",
                (concat_ws(";", *payment_profits_primary_key_fields).cast("string")),
            )
        )

        payment_profits_df = payment_profits_df.select(
            payment_profits_df.PaymentId,
            lit("profits").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            payment_profits_df.PaymentKey,
            payment_profits_df.PaymentUniqueNumber,
            payment_profits_df.ACCOUNT_NUMBER.alias("ExposureKey"),
            payment_profits_df.actual_date.cast("Date").alias("PaymentActualDate"),
            payment_profits_df.Record_date.cast("Timestamp").alias("PaymentRecordDate"),
            payment_profits_df.amount.cast("Decimal(15,2)").alias("PaymentAmount"),
            payment_profits_df.PaymentIntIdentification,
            lit(None).alias("PaymentCostAmount"),
            lit(None).alias("CoverageAmount"),
            lit(None).alias("DebtAmount"),
            lit(None).alias("MinimumPayment"),
            payment_profits_df.CURRENCY.alias("CurrencyCode"),
            payment_profits_df.CURRENCY.alias("Currency"),
            payment_profits_df.GD_REASON.cast("string").alias("PaymentTypeCode"),
            params.description.cast("string").alias("PaymentType"),
            lit(None).alias("ReasonCode"),
            lit(None).alias("TransactionReasonCode"),
            lit(None).alias("TransactionReason"),
            lit(None).alias("ReportingDate"),
            payment_profits_df.principal.cast("Decimal(15,2)").alias("PrincipalAmount"),
            payment_profits_df.commission.cast("Decimal(15,2)").alias(
                "CommissionAmount"
            ),
            payment_profits_df.interest.cast("Decimal(15,2)").alias("InterestAmount"),
            payment_profits_df.expenses.cast("Decimal(15,2)").alias("ExpensesAmount"),
            payment_profits_df.INTEREST_URL.cast("string").alias("InterestURL"),
            payment_profits_df.trx_usr.alias("Createdby"),
            payment_profits_df.isdownpayment.alias("IsDownPaymentFlag"),
            payment_profits_df.REVERSED_FLG.alias("IsReversedFlag"),
            lit(None).alias("PaymentISCOL"),
            lit(None).alias("PaymentRecordType"),
            paymenttypegroupmapping.PaymentTypeGroup.alias("PaymentTypeGroup"),
            paymenttypegroupmapping.ProfitsPaymentType.alias("PaymentTypeEng"),
            payments.businessdate.alias("BusinessDate"),
            
        )
        # ================== Append ==================

        payment_profits_df.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_payment_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)